In [0]:
# Install Dependencies
%pip install faker --quiet
dbutils.library.restartPython()

In [0]:
# Imports & Global Config
import random
import json
import os
import gc
import uuid
import math
from datetime import date, datetime, timedelta

import pandas as pd
import numpy as np
from faker import Faker

In [0]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('de_DE')
Faker.seed(SEED)

In [0]:
# DATE RANGE - 6 month simulation 
START_DATE = date(2026, 4, 1)
END_DATE   = date(2026, 10, 1)
TOTAL_DAYS = (END_DATE - START_DATE).days + 1

In [0]:
TRIPS_PER_DAY_BASE      = 35_000   # Scalable to 120k for full stress tests
EVENTS_PER_DAY_BASE     = 70_000
TOTAL_CUSTOMERS         = 50_000
TOTAL_DRIVERS           = 8_000
NEW_CUSTOMERS_PER_DAY   = 300

In [0]:
CATALOG   = 'fahrlytics'
BRONZE    = f'{CATALOG}.bronze'
SILVER    = f'{CATALOG}.silver'
GOLD      = f'{CATALOG}.gold'
BASE_PATH = f'/Volumes/{CATALOG}/bronze/raw_data'

In [0]:
# ── CITY CONFIGURATION (GERMANY) ─────────────────────────────────────────
CITIES       = ['Berlin', 'Munich', 'Hamburg', 'Frankfurt', 'Cologne']
CITY_WEIGHTS = [0.30, 0.25, 0.20, 0.15, 0.10]
CITY_CHURN   = {
    'Berlin': 0.16,
    'Munich': 0.14,
    'Hamburg': 0.18,
    'Frankfurt': 0.22,
    'Cologne': 0.25
}

# Real-world dirty name variations for silver data quality cleaning
DIRTY_CITY = {
    'Berlin':    ['berlin', 'Berlin-Mitte', 'Berln', 'Berlin'],
    'Munich':    ['München', 'Munich', 'muenchen', 'München-Stadt'],
    'Hamburg':   ['hamburg', 'HH', 'Hamburg-Nord', 'Hamburg'],
    'Frankfurt': ['Frankfurt am Main', 'frankfurt', 'FFM', 'Frankfurt/M'],
    'Cologne':   ['Köln', 'Koeln', 'cologne', 'Cologne'],
}

In [0]:
# ── GERMAN PUBLIC HOLIDAYS & MAJOR EVENTS (FEIERTAGE) ───────────────────
def get_german_holidays():
    """German national public holidays and high-demand festive dates."""
    return {
        date(2026, 4, 3):  'Karfreitag',
        date(2026, 4, 6):  'Ostermontag',
        date(2026, 5, 1):  'Tag der Arbeit',
        date(2026, 5, 14): 'Christi Himmelfahrt / Vatertag',
        date(2026, 5, 25): 'Pfingstmontag',
        date(2026, 6, 4):  'Fronleichnam',
        date(2026, 10, 3): 'Tag der Deutschen Einheit',
        # Late-night / festival boosts
        date(2026, 9, 19): 'Oktoberfest Opening (Munich surge)',
    }

HOLIDAYS = get_german_holidays()

def is_weekend(d): 
    return d.weekday() >= 5  # Sat=5, Sun=6

def is_holiday(d): 
    return d in HOLIDAYS

def day_demand_multiplier(d):
    """Calibrate ride demand by holiday and day of week."""
    if d in HOLIDAYS:
        return 1.35
    if d.weekday() == 4:  # Friday evening surge
        return 1.20
    if d.weekday() == 5:  # Saturday nightlife
        return 1.15
    if d.weekday() == 6:  # Sunday
        return 0.80
    return 1.0  # Regular weekday baseline

print(f'Project Catalog : {CATALOG}')
print(f'Landing Volume  : {BASE_PATH}')
print(f'Date Range      : {START_DATE} to {END_DATE} ({TOTAL_DAYS} days)')
print(f'Holidays Loaded : {len(HOLIDAYS)}')

In [0]:
# Setup Catalog Schemas & Landing Directories
print('Verifying Unity Catalog and Volume structure...')

spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'USE CATALOG {CATALOG}')

for schema in ['bronze', 'silver', 'gold']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')

# Create raw_data Volume under Bronze
spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw_data')

# Create directory branches inside the volume
dirs = [
    f'{BASE_PATH}/customers',
    f'{BASE_PATH}/drivers',
    f'{BASE_PATH}/trips',
    f'{BASE_PATH}/payments',
    f'{BASE_PATH}/clickstream'
]
for d in dirs:
    dbutils.fs.mkdirs(d)

print('Volume folder hierarchy confirmed.')

In [0]:
# Helper Functions & Domain Logic
def get_surge(hour, weather):
    """Calculates surge pricing based on commuting hours and weather."""
    if 7 <= hour <= 9:     # Morning rush
        surge = random.choices([1.0, 1.3, 1.8], weights=[0.45, 0.35, 0.20])[0]
    elif 16 <= hour <= 19:  # Evening rush
        surge = random.choices([1.0, 1.4, 2.0], weights=[0.40, 0.40, 0.20])[0]
    elif 23 <= hour or hour <= 4: # Late night
        surge = random.choices([1.0, 1.5, 2.2], weights=[0.50, 0.30, 0.20])[0]
    else:
        surge = random.choices([1.0, 1.2], weights=[0.85, 0.15])[0]
        
    if weather == 'Rain': 
        surge = min(surge + 0.3, 2.5)
    return round(surge, 2)

def get_weather(city, month):
    """Central European seasonal weather distributions."""
    if month in [4, 5]: # Spring
        return random.choices(['Clear', 'Rain', 'Windy'], weights=[0.55, 0.35, 0.10])[0]
    elif month in [6, 7, 8]: # Summer
        return random.choices(['Clear', 'Rain', 'Thunderstorm'], weights=[0.70, 0.25, 0.05])[0]
    return random.choices(['Clear', 'Rain', 'Fog'], weights=[0.50, 0.40, 0.10])[0]

def calc_fare(distance_km, surge, vehicle_type):
    """Standard German taxi/ride-hailing tariffs in EUR (€)."""
    base = {'Standard': 4.20, 'Comfort': 6.00, 'Van/XL': 8.50}
    rate = {'Standard': 2.10, 'Comfort': 2.75, 'Van/XL': 3.40}
    fare = base[vehicle_type] + (rate[vehicle_type] * distance_km)
    fare *= surge
    return round(max(5.00, fare + random.uniform(-1.5, 1.5)), 2)

def dirty_city(city):
    """Embeds dirty entity representations at ~15% probability."""
    if random.random() < 0.15:
        return random.choice(DIRTY_CITY.get(city, [city]))
    return city

In [0]:
# Generate Drivers Master File
print('Generating drivers master dataset...')

vehicle_types = ['Standard', 'Comfort', 'Van/XL']
v_weights     = [0.65, 0.25, 0.10]

drivers = []
for _ in range(TOTAL_DRIVERS):
    city = random.choices(CITIES, weights=CITY_WEIGHTS)[0]
    vtype = random.choices(vehicle_types, weights=v_weights)[0]
    status = 'active' if random.random() < 0.94 else 'inactive'
    join_days_ago = random.randint(180, 1200)
    join_date = (START_DATE - timedelta(days=join_days_ago)).isoformat()
    
    drivers.append({
        'driver_id': str(uuid.uuid4()),
        'name': fake.name(),
        'city': city,
        'vehicle_type': vtype,
        'rating': round(random.uniform(4.0, 5.0), 2),
        'trips_completed': random.randint(50, 6500),
        'joining_date': join_date,
        'status': status,
    })

driver_df = pd.DataFrame(drivers)
driver_df.to_csv(f'{BASE_PATH}/drivers/drivers_master.csv', index=False)

active_drivers = driver_df[driver_df['status'] == 'active'][['driver_id', 'city', 'vehicle_type']].to_dict('records')
driver_by_city = {c: [d for d in active_drivers if d['city'] == c] for c in CITIES}
print(f'drivers_master.csv generated: {len(drivers):,} records ({len(active_drivers):,} active).')

In [0]:
# Generate Customers (Daily Snapshots)
print('Generating daily customer snapshots...')

all_customer_ids = []
churned_map = {}
customer_city_map = {}
customer_device_map = {}
customer_ab_group = {}
total_cust_written = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')
    
    n_new = int(NEW_CUSTOMERS_PER_DAY * day_demand_multiplier(current_date))
    n_new = max(100, n_new + random.randint(-30, 30))
    
    rows = []
    for _ in range(n_new):
        city = random.choices(CITIES, weights=CITY_WEIGHTS)[0]
        device = random.choices(['iOS', 'Android', 'Web'], weights=[0.52, 0.43, 0.05])[0]
        ref_src = random.choices(['Direct/Organic', 'Performance Ads', 'Friend Referral'], weights=[0.45, 0.35, 0.20])[0]
        cid = str(uuid.uuid4())
        
        # Churn probability
        churn_rate = CITY_CHURN[city]
        is_churned = random.random() < churn_rate
        churn_dt = None
        if is_churned:
            days_till_churn = random.randint(20, 100)
            churn_dt = current_date + timedelta(days=days_till_churn)
            if churn_dt > END_DATE:
                churn_dt = None
                is_churned = False
                
        # Data quality injections
        rating = round(random.uniform(3.8, 5.0), 1)
        if random.random() < 0.005: 
            rating = round(random.uniform(5.1, 7.5), 1) # Invalid rating
        elif random.random() < 0.005:
            rating = None # Null rating
            
        row = {
            'customer_id': cid,
            'name': fake.name(),
            'city': dirty_city(city),
            'signup_date': current_date.isoformat(),
            'device_type': device,
            'referral_source': ref_src,
            'rating': rating,
            'is_churned': is_churned,
            'churn_date': churn_dt.isoformat() if churn_dt else None
        }
        
        rows.append(row)
        # Duplicate row injection (~1%)
        if random.random() < 0.01:
            rows.append(row.copy())
            
        all_customer_ids.append(cid)
        customer_city_map[cid] = city
        customer_device_map[cid] = device
        customer_ab_group[cid] = 'variant' if hash(cid) % 2 == 0 else 'control'
        if is_churned and churn_dt:
            churned_map[cid] = churn_dt
            
    fpath = f'{BASE_PATH}/customers/customers_{ds}.csv'
    pd.DataFrame(rows).to_csv(fpath, index=False)
    total_cust_written += len(rows)

print(f'Customer datasets generated: {total_cust_written:,} records written across {TOTAL_DAYS} days.')

In [0]:
# Generate Trips (Daily JSON Files)
print('Generating daily trip logs...')

trip_ids_by_date = {}
trip_fare_by_id = {}
trip_cust_by_id = {}
total_trips = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')
    
    multiplier = day_demand_multiplier(current_date)
    n_trips = int(TRIPS_PER_DAY_BASE * multiplier) + random.randint(-500, 500)
    
    # Filter active population eligible to book
    eligible_customers = [
        cid for cid in all_customer_ids
        if cid not in churned_map or churned_map[cid] > current_date
    ]
    if len(eligible_customers) < 500:
        eligible_customers = all_customer_ids[:500]
        
    rows = []
    completed_trips = []
    
    for _ in range(n_trips):
        cid = random.choice(eligible_customers)
        home_city = customer_city_map.get(cid, 'Berlin')
        trip_city = home_city if random.random() < 0.85 else random.choice(CITIES)
        
        city_drivers = driver_by_city.get(trip_city, active_drivers)
        driver = random.choice(city_drivers) if city_drivers else random.choice(active_drivers)
        
        # Hourly distribution
        hour = random.choices(
            range(24),
            weights=[1,1,1,1,1,2,4,7,9,6,5,5,5,5,5,6,8,9,8,6,5,4,3,2]
        )[0]
        minute = random.randint(0, 59)
        pickup_dt = datetime(current_date.year, current_date.month, current_date.day, hour, minute)
        duration_min = random.randint(8, 45)
        drop_dt = pickup_dt + timedelta(minutes=duration_min)
        
        weather = get_weather(trip_city, current_date.month)
        surge = get_surge(hour, weather)
        
        # Distance anomalies: Negative values (~0.3%)
        distance = round(max(0.8, np.random.normal(6.5, 3.2)), 2)
        if random.random() < 0.003:
            distance = -round(random.uniform(0.5, 4.0), 2)
            
        # Ghost trip fraud injection (~0.2%)
        is_ghost = random.random() < 0.002
        if is_ghost:
            distance = 0.0
            
        fare = calc_fare(abs(distance), surge, driver['vehicle_type'])
        
        # Fare inflation anomaly (~0.1%)
        is_inflated = False
        if random.random() < 0.001:
            fare *= random.uniform(2.5, 3.8)
            is_inflated = True
            
        status = random.choices(['completed', 'cancelled'], weights=[0.92, 0.08])[0]
        trip_id = str(uuid.uuid4())
        
        row = {
            'trip_id': trip_id,
            'customer_id': cid,
            'driver_id': driver['driver_id'],
            'city': trip_city,
            'pickup_time': pickup_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'drop_time': drop_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'distance_km': distance,
            'fare_amount': round(fare, 2),
            'currency': 'EUR',
            'surge_multiplier': surge,
            'trip_status': status,
            'vehicle_type': driver['vehicle_type'],
            'weather': weather,
            'is_ghost_trip': is_ghost,
            'is_fare_inflated': is_inflated,
            'day_type': 'holiday' if is_holiday(current_date) else ('weekend' if is_weekend(current_date) else 'weekday')
        }
        rows.append(row)
        
        if status == 'completed':
            completed_trips.append(trip_id)
            trip_fare_by_id[trip_id] = fare
            trip_cust_by_id[trip_id] = cid
            
    trip_ids_by_date[current_date] = completed_trips
    total_trips += len(rows)
    
    fpath = f'{BASE_PATH}/trips/trips_{ds}.json'
    with open(fpath, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

print(f'Trips datasets generated: {total_trips:,} trips logged.')

In [0]:
# Generate Payments (Daily CSV Files)
print('Generating payment logs...')

# Coordinated fraud cards across accounts
n_fraud_cards = max(10, int(len(all_customer_ids) * 0.001))
fraud_cards = [f'DE_CC_{str(uuid.uuid4())[:12]}' for _ in range(n_fraud_cards)]
cust_to_fraud_card = {}
for card in fraud_cards:
    linked_custs = random.sample(all_customer_ids, min(4, len(all_customer_ids)))
    for c in linked_custs:
        cust_to_fraud_card[c] = card

total_payments = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')
    completed = trip_ids_by_date.get(current_date, [])
    
    rows = []
    for tid in completed:
        fare = trip_fare_by_id.get(tid, 18.50)
        cid = trip_cust_by_id.get(tid, random.choice(all_customer_ids))
        
        method = random.choices(
            ['PayPal', 'Credit Card', 'Apple Pay / Google Pay', 'EC-Karte'],
            weights=[0.45, 0.30, 0.15, 0.10]
        )[0]
        
        card_hash = None
        if method == 'Credit Card':
            card_hash = cust_to_fraud_card.get(cid, f'DE_CC_{str(uuid.uuid4())[:12]}')
            
        amount = fare
        # Fare mismatch anomaly (~0.2%)
        if random.random() < 0.002:
            amount = round(fare * random.uniform(0.6, 1.5), 2)
            
        pay_offset = random.randint(10, 360)
        pay_status = random.choices(['success', 'failed'], weights=[0.975, 0.025])[0]
        
        rows.append({
            'payment_id': str(uuid.uuid4()),
            'trip_id': tid,
            'customer_id': cid,
            'amount': round(amount, 2),
            'currency': 'EUR',
            'payment_method': method,
            'payment_timestamp': f'{current_date.isoformat()} 00:{pay_offset//60:02}:{pay_offset%60:02}',
            'payment_status': pay_status,
            'card_hash': card_hash,
            'is_fraud_card': cid in cust_to_fraud_card
        })
        
    # Orphan payments (~0.25% with invalid trip_id)
    n_orphans = max(1, int(len(rows) * 0.0025))
    for _ in range(n_orphans):
        rows.append({
            'payment_id': str(uuid.uuid4()),
            'trip_id': str(uuid.uuid4()),
            'customer_id': random.choice(all_customer_ids),
            'amount': round(random.uniform(10.0, 45.0), 2),
            'currency': 'EUR',
            'payment_method': 'PayPal',
            'payment_timestamp': current_date.isoformat(),
            'payment_status': 'success',
            'card_hash': None,
            'is_fraud_card': False
        })
        
    fpath = f'{BASE_PATH}/payments/payments_{ds}.csv'
    pd.DataFrame(rows).to_csv(fpath, index=False)
    total_payments += len(rows)

print(f'Payments datasets generated: {total_payments:,} transaction records.')

In [0]:
# Generate Clickstream Events (Daily JSON Files)
print('Generating clickstream telemetry...')

total_events = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')
    
    multiplier = day_demand_multiplier(current_date)
    n_events = int(EVENTS_PER_DAY_BASE * multiplier) + random.randint(-1500, 1500)
    
    active_customers = [
        cid for cid in all_customer_ids
        if cid not in churned_map or churned_map[cid] > current_date
    ]
    if len(active_customers) < 300:
        active_customers = all_customer_ids[:300]
        
    rows = []
    n_sessions = n_events // 4
    
    for _ in range(n_sessions):
        cid = random.choice(active_customers)
        city = customer_city_map.get(cid, 'Berlin')
        device = customer_device_map.get(cid, 'iOS')
        ab_group = customer_ab_group.get(cid, 'control')
        session_id = str(uuid.uuid4())
        
        hour = random.randint(6, 23)
        base_ts = datetime(current_date.year, current_date.month, current_date.day, hour, random.randint(0, 59))
        
        # A/B test variant: dynamic pricing screen experiment
        confirm_weight = 0.13 if ab_group == 'variant' else 0.10
        cancel_weight  = 0.03 if ab_group == 'variant' else 0.05
        
        session_flow = ['app_open']
        if random.random() < 0.70: 
            session_flow.append('search_ride')
        if random.random() < 0.65 and len(session_flow) > 1: 
            session_flow.append('view_fare_quote')
            
        r = random.random()
        if r < confirm_weight and len(session_flow) > 2: 
            session_flow.append('confirm_booking')
        elif r < confirm_weight + cancel_weight: 
            session_flow.append('cancel_booking')
            
        for seq, etype in enumerate(session_flow):
            ts = base_ts + timedelta(seconds=seq * random.randint(5, 45))
            rows.append({
                'event_id': str(uuid.uuid4()),
                'customer_id': cid,
                'session_id': session_id,
                'event_type': etype,
                'event_timestamp': ts.strftime('%Y-%m-%d %H:%M:%S'),
                'city': city,
                'device_type': device,
                'ab_test_group': ab_group,
                'ab_test_experiment': 'dynamic_pricing_de_v1'
            })
            
    fpath = f'{BASE_PATH}/clickstream/events_{ds}.json'
    with open(fpath, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
            
    total_events += len(rows)

print(f'Clickstream generation completed: {total_events:,} events logged.')